<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/03_rev_and_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 3 of 7: Representative Elementary Volume (REV) and Uncertainty

*OpenImpala Tutorial Series — From first solve to HPC deployment*

---

When you compute the tortuosity of a single crop from a micro-CT scan, the result depends on where and how large that crop is. A natural question is: *at what sample size do the computed properties stabilise?* This is the concept of the **Representative Elementary Volume (REV)**.

In this tutorial we take a rigorous statistical approach — computing the tortuosity of many random sub-volumes at different sizes — to identify the REV, quantify uncertainty, and understand when common assumptions (like normality) break down.

**What you will learn:**
1. Extract random sub-volumes of varying sizes from a 3D image.
2. Solve the transport problem for each crop (over 100 simulations).
3. Plot a violin-style REV diagram showing the full distribution at each scale.
4. Use the Coefficient of Variation (CV) to define the REV threshold.
5. Track **percolation probability** — how often sub-volumes fail to span the domain.
6. Compute **bootstrap confidence intervals** for the mean tortuosity at each scale.
7. **Test normality** with Q-Q plots and the Shapiro-Wilk test to know when Gaussian error bars are valid.
8. Compare the REV across **different transport directions** (anisotropic REV).
9. Produce a **publication-ready uncertainty summary** — what to report in a paper.

**Prerequisites:** [Tutorial 1](01_hello_openimpala.ipynb) — core OpenImpala workflow. Familiarity with [Tutorial 2](02_digital_twin.ipynb) (loading real images, directional tortuosity) is helpful but not required.

In [ ]:
# Install OpenImpala and utilities
!pip install -q openimpala tifffile matplotlib scipy

In [ ]:
import os
import time
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded.")

# In a notebook we open the session manually so it stays alive across
# cells.  In a script, use `with oi.Session():` instead (see Tutorial 1).
session = oi.Session()
session.__enter__()

# Download the sample TIFF so the notebook works on Colab.
data_url = "https://raw.githubusercontent.com/BASE-Laboratory/OpenImpala/master/data/SampleData_2Phase_stack_3d_1bit.tif"
data_path = "SampleData_2Phase_stack_3d_1bit.tif"

if not os.path.exists(data_path):
    print("Downloading sample data...")
    urllib.request.urlretrieve(data_url, data_path)
    print("Download complete.")

# Load the scan into memory
microstructure = tifffile.imread(data_path).astype(np.int32)
print(f"Base microstructure loaded: {microstructure.shape}")

## 1. Setting Up the Statistical Sampling

If you pick a small sub-volume (say $16^3$), the tortuosity will vary substantially depending on whether you happen to sample a region with large pores or dense solid. As the sub-volume size increases, these fluctuations should average out and the computed properties should converge.

We define a set of window sizes. For each size, we randomly crop the microstructure 20 times and record:
- Whether the pore phase **percolates** (spans inlet to outlet).
- The **local porosity** and **tortuosity** for every percolating crop.

Tracking percolation failures is important — at small scales, a sub-volume may contain an isolated pore cluster that never connects to the boundary, making the tortuosity undefined. The fraction of samples that percolate is itself a meaningful statistic.

In [ ]:
window_sizes = [16, 24, 32, 40, 48, 64, 80]
n_samples_per_size = 20  # 140 simulations in total
total_evals = len(window_sizes) * n_samples_per_size

rev_results = {w: {'tau': [], 'vf': [], 'n_total': 0, 'n_percolating': 0} for w in window_sizes}

print(f"Running {total_evals} simulations (this may take a few minutes)...")
t_start = time.time()
total_solves = 0
eval_count = 0

for w in window_sizes:
    max_start_idx = microstructure.shape[0] - w

    for _ in range(n_samples_per_size):
        eval_count += 1

        # 1. Random crop origin
        z = np.random.randint(0, max_start_idx + 1)
        y = np.random.randint(0, max_start_idx + 1)
        x = np.random.randint(0, max_start_idx + 1)

        # 2. Extract the sub-volume
        crop = microstructure[z:z+w, y:y+w, x:x+w]

        # 3. Local porosity
        vf = oi.volume_fraction(crop, phase=1).fraction

        # 4. Percolation check
        rev_results[w]['n_total'] += 1
        perc = oi.percolation_check(crop, phase=1, direction="z")

        # 5. Solve if percolating
        if perc.percolates:
            rev_results[w]['n_percolating'] += 1
            res = oi.tortuosity(crop, phase=1, direction="z", solver="flexgmres")
            rev_results[w]['tau'].append(res.tortuosity)
            rev_results[w]['vf'].append(vf)
            total_solves += 1

        # Progress indicator every 20 evaluations
        if eval_count % 20 == 0:
            elapsed = time.time() - t_start
            rate = eval_count / elapsed
            remaining = (total_evals - eval_count) / rate
            print(f"  [{eval_count}/{total_evals}] {elapsed:.0f}s elapsed, "
                  f"~{remaining:.0f}s remaining ({total_solves} solves so far)")

t_end = time.time()
print(f"\nCompleted {total_solves} solves in {t_end - t_start:.1f}s.")

# Quick summary of percolation rates
for w in window_sizes:
    r = rev_results[w]
    perc_pct = 100 * r['n_percolating'] / r['n_total']
    n_tau = len(r['tau'])
    print(f"  {w:3d}^3: {r['n_percolating']}/{r['n_total']} percolated ({perc_pct:.0f}%), "
          f"{n_tau} tortuosity values")

## 2. REV Violin Diagram

A mean and standard deviation assume the data are normally distributed. A **violin plot** shows the full probability density at each scale, which is more informative — particularly at small sizes where the distribution can be strongly skewed (long tails of high-tortuosity bottleneck regions).

In [ ]:
means = []
stds = []
valid_sizes = []
plot_data = []

# Extract data, ignoring sizes where nothing percolated
for w in window_sizes:
    taus = rev_results[w]['tau']
    if len(taus) > 0:
        means.append(np.mean(taus))
        stds.append(np.std(taus))
        valid_sizes.append(w)
        plot_data.append(taus)

means = np.array(means)
stds = np.array(stds)
valid_sizes = np.array(valid_sizes)

fig, ax = plt.subplots(figsize=(9, 5), dpi=150)

# Render the Violin Plot
parts = ax.violinplot(plot_data, positions=valid_sizes, widths=5,
                      showmeans=False, showextrema=False)

# Style the violins
for pc in parts['bodies']:
    pc.set_facecolor('#1f77b4')
    pc.set_edgecolor('black')
    pc.set_alpha(0.5)

# Plot the Mean trend line
ax.plot(valid_sizes, means, color='#d62728', linewidth=2.5, marker='o', markersize=6, label='Mean Tortuosity')

ax.set_xlabel("Window Size (Voxels)", fontsize=12, fontweight='bold')
ax.set_ylabel(r"Tortuosity Factor ($\tau$)", fontsize=12, fontweight='bold')
ax.set_title("Probability Density of Microstructural Transport (REV)", fontsize=14, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend()

plt.tight_layout()
plt.show()

## 3. Quantifying Uncertainty with the Coefficient of Variation

To determine whether a given sample size is representative, we use the **Coefficient of Variation**: $\mathrm{CV} = \sigma / \mu \times 100\%$. A common practical threshold is CV < 5%.

The right-hand plot below examines *why* the variance is large at small scales: it shows the correlation between local porosity and local tortuosity for the $32^3$ sub-volumes. In a small crop, the porosity itself fluctuates, and tortuosity responds accordingly.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

# --- SUBPLOT 1: Coefficient of Variation ---
cvs = (stds / means) * 100
ax1.plot(valid_sizes, cvs, color='#2ca02c', linewidth=2.5, marker='s', markersize=7)
ax1.axhline(y=5.0, color='r', linestyle='--', linewidth=2, label='5% Threshold (REV Limit)')

ax1.set_xlabel("Window Size (Voxels)", fontweight='bold')
ax1.set_ylabel("Coefficient of Variation (%)", fontweight='bold')
ax1.set_title("Defining the REV Threshold", fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- SUBPLOT 2: The Physical Origin of Variance ---
# Let's look at the sub-volumes from the W=32 window size
w_small = 32
local_vfs = np.array(rev_results[w_small]['vf']) * 100  # Convert to percentage
local_taus = np.array(rev_results[w_small]['tau'])

ax2.scatter(local_vfs, local_taus, color='#ff7f0e', edgecolor='black', s=60, alpha=0.8)

# Add a linear trendline
if len(local_vfs) > 1:
    z = np.polyfit(local_vfs, local_taus, 1)
    p = np.poly1d(z)
    ax2.plot(local_vfs, p(local_vfs), "k-", alpha=0.6, linewidth=2)

ax2.set_xlabel("Local Porosity (%)", fontweight='bold')
ax2.set_ylabel(r"Local Tortuosity ($\tau$)", fontweight='bold')
ax2.set_title(f"Physical Correlation (Scale: {w_small}^3)", fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Percolation Probability

Before a sub-volume can have a well-defined tortuosity, the transport phase must percolate — it must form a connected path from the inlet face to the outlet face. At small scales, some crops will contain isolated pore clusters that do not span the domain.

The **percolation probability** $P_\text{perc}(L)$ — the fraction of sub-volumes of side length $L$ that percolate — is itself a scale-dependent property. It should converge to 1.0 (or 0.0 for non-percolating phases) as $L$ approaches the REV. If $P_\text{perc} < 1$ at your chosen sample size, your tortuosity statistics are conditioned on percolation and may be biased (you are missing the hardest-to-transport regions).

In [ ]:
perc_probs = []
perc_sizes = []

for w in window_sizes:
    r = rev_results[w]
    if r['n_total'] > 0:
        perc_probs.append(100 * r['n_percolating'] / r['n_total'])
        perc_sizes.append(w)

fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
ax.plot(perc_sizes, perc_probs, 'o-', color='#9467bd', linewidth=2.5, markersize=8,
        markerfacecolor='white', markeredgewidth=2)
ax.axhline(y=100, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel("Window Size (Voxels)", fontweight='bold')
ax.set_ylabel("Percolation Probability (%)", fontweight='bold')
ax.set_title("Fraction of Sub-Volumes with Connected Pore Phase", fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(True, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Annotate the first size where percolation reaches 100%
for i, (s, p) in enumerate(zip(perc_sizes, perc_probs)):
    if p >= 100:
        ax.annotate(f'100% at L={s}', xy=(s, p), xytext=(s + 5, p - 10),
                    fontsize=10, fontweight='bold', color='#9467bd',
                    arrowprops=dict(arrowstyle='->', color='#9467bd'))
        break

plt.tight_layout()
plt.show()

# Warn the user about conditioning bias
for w in window_sizes:
    r = rev_results[w]
    if r['n_percolating'] < r['n_total']:
        pct = 100 * r['n_percolating'] / r['n_total']
        print(f"  Warning: At L={w}, only {pct:.0f}% of sub-volumes percolate. "
              f"Tortuosity statistics at this scale are conditioned on percolation.")

## 5. Bootstrap Confidence Intervals

The Coefficient of Variation gives a single-number summary of spread, but for publication you typically need **confidence intervals** for the mean. The bootstrap is ideal here because it makes no assumption about the underlying distribution — it works whether the data are Gaussian, skewed, or heavy-tailed.

The procedure:
1. From the $n$ tortuosity values at a given window size, draw $B = 10{,}000$ bootstrap samples (sampling with replacement).
2. Compute the mean of each bootstrap sample.
3. Take the 2.5th and 97.5th percentiles of the bootstrap means as the 95% CI.

This is a **nonparametric** interval — unlike $\bar{\tau} \pm 1.96\,\sigma/\sqrt{n}$, it does not assume normality.

In [ ]:
B = 10_000  # Number of bootstrap resamples
rng = np.random.default_rng(42)

boot_ci_lo = []
boot_ci_hi = []
boot_means = []
gaussian_ci_lo = []
gaussian_ci_hi = []

for i, w in enumerate(valid_sizes):
    taus = np.array(plot_data[i])
    n = len(taus)

    # Bootstrap 95% CI for the mean
    boot_samples = rng.choice(taus, size=(B, n), replace=True)
    boot_sample_means = boot_samples.mean(axis=1)
    ci_lo, ci_hi = np.percentile(boot_sample_means, [2.5, 97.5])
    boot_ci_lo.append(ci_lo)
    boot_ci_hi.append(ci_hi)
    boot_means.append(np.mean(taus))

    # Gaussian CI for comparison: mean +/- 1.96 * SEM
    sem = np.std(taus, ddof=1) / np.sqrt(n)
    gaussian_ci_lo.append(np.mean(taus) - 1.96 * sem)
    gaussian_ci_hi.append(np.mean(taus) + 1.96 * sem)

boot_means = np.array(boot_means)
boot_ci_lo = np.array(boot_ci_lo)
boot_ci_hi = np.array(boot_ci_hi)
gaussian_ci_lo = np.array(gaussian_ci_lo)
gaussian_ci_hi = np.array(gaussian_ci_hi)

# --- Plot: Bootstrap vs Gaussian CIs ---
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)

# Bootstrap CI (shaded band)
ax.fill_between(valid_sizes, boot_ci_lo, boot_ci_hi,
                alpha=0.3, color='#1f77b4', label='Bootstrap 95% CI')

# Gaussian CI (dashed lines)
ax.plot(valid_sizes, gaussian_ci_lo, '--', color='#ff7f0e', linewidth=1.5)
ax.plot(valid_sizes, gaussian_ci_hi, '--', color='#ff7f0e', linewidth=1.5,
        label=r'Gaussian 95% CI ($\bar{\tau} \pm 1.96\,\mathrm{SEM}$)')

# Mean
ax.plot(valid_sizes, boot_means, 'o-', color='#d62728', linewidth=2.5,
        markersize=7, label='Sample Mean')

ax.set_xlabel("Window Size (Voxels)", fontweight='bold')
ax.set_ylabel(r"Tortuosity Factor ($\tau$)", fontweight='bold')
ax.set_title("Confidence Intervals for the Mean Tortuosity", fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Print the CI widths
print("CI width comparison (Bootstrap vs Gaussian):")
print(f"{'Size':>6s}  {'Mean':>7s}  {'Boot CI':>18s}  {'Boot Width':>10s}  {'Gauss Width':>11s}")
for i, w in enumerate(valid_sizes):
    bw = boot_ci_hi[i] - boot_ci_lo[i]
    gw = gaussian_ci_hi[i] - gaussian_ci_lo[i]
    print(f"{int(w):6d}  {boot_means[i]:7.4f}  "
          f"[{boot_ci_lo[i]:.4f}, {boot_ci_hi[i]:.4f}]  "
          f"{bw:10.4f}  {gw:11.4f}")

## 6. Testing Distributional Assumptions

Many experimentalists report tortuosity as $\bar{\tau} \pm \sigma$ and implicitly assume the data are normally distributed. But is that assumption valid?

At small scales the tortuosity distribution is often **right-skewed**: most sub-volumes have moderate tortuosity, but a few contain bottleneck regions with very high values. As the window size increases and the distribution converges, it typically becomes more symmetric and better approximated by a Gaussian.

We test this in two ways:
- **Q-Q plot** — plots the sample quantiles against the theoretical quantiles of a normal distribution. If the data are Gaussian, the points fall on the diagonal.
- **Shapiro-Wilk test** — a formal hypothesis test for normality. A p-value below 0.05 rejects the null hypothesis that the data are normally distributed.

Understanding when normality holds (and when it doesn't) tells you whether standard error bars ($\pm 1.96\,\sigma/\sqrt{n}$) are appropriate, or whether you need the bootstrap CIs from Section 5.

In [ ]:
from scipy import stats

# Pick two contrasting scales: the smallest and largest with enough data
small_idx = 0   # smallest valid window size
large_idx = -1  # largest valid window size

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

for ax, idx, color in zip(axes, [small_idx, large_idx], ['#ff7f0e', '#1f77b4']):
    taus = np.array(plot_data[idx])
    w = int(valid_sizes[idx])

    # Q-Q plot
    (osm, osr), (slope, intercept, r) = stats.probplot(taus, dist="norm")
    ax.scatter(osm, osr, color=color, edgecolor='black', s=50, alpha=0.8, zorder=3)
    qq_line_x = np.array([osm.min(), osm.max()])
    ax.plot(qq_line_x, slope * qq_line_x + intercept, 'r--', lw=2)

    # Shapiro-Wilk test
    if len(taus) >= 3:
        stat_sw, p_sw = stats.shapiro(taus)
        verdict = "Cannot reject normality" if p_sw >= 0.05 else "Normality rejected"
        ax.set_title(f"Q-Q Plot: L = {w}$^3$\nShapiro-Wilk p = {p_sw:.3f} ({verdict})",
                     fontweight='bold', fontsize=11)
    else:
        ax.set_title(f"Q-Q Plot: L = {w}$^3$ (too few samples)", fontweight='bold')

    ax.set_xlabel("Theoretical Quantiles (Normal)", fontweight='bold')
    ax.set_ylabel("Sample Quantiles (Tortuosity)", fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Run Shapiro-Wilk for all window sizes and print a summary table
print("\nShapiro-Wilk normality test across all scales:")
print(f"{'Size':>6s}  {'N':>4s}  {'W-stat':>8s}  {'p-value':>8s}  {'Skewness':>9s}  {'Normal?':>10s}")
print("-" * 55)
for i, w in enumerate(valid_sizes):
    taus = np.array(plot_data[i])
    n = len(taus)
    if n >= 3:
        stat_sw, p_sw = stats.shapiro(taus)
        skew = stats.skew(taus)
        normal = "Yes" if p_sw >= 0.05 else "No"
        print(f"{int(w):6d}  {n:4d}  {stat_sw:8.4f}  {p_sw:8.4f}  {skew:9.3f}  {normal:>10s}")
    else:
        print(f"{int(w):6d}  {n:4d}  {'N/A':>8s}  {'N/A':>8s}  {'N/A':>9s}  {'too few':>10s}")

## 7. Directional REV: Does the REV Size Depend on Direction?

In [Tutorial 2](02_digital_twin.ipynb) we saw that tortuosity can be anisotropic — different in X, Y, and Z. A less obvious but equally important question is whether the **REV size itself** is anisotropic. A material that is calendered (compressed) in the Z-direction may have a larger REV for Z-transport than for X- or Y-transport, because the critical pore bottlenecks that dominate Z-tortuosity are rarer and require a larger sample to average over.

Below we repeat the REV analysis for a subset of window sizes in all three directions and compare the CV convergence curves.

In [ ]:
# Use a subset of sizes to keep runtime manageable
dir_sizes = [24, 40, 64, 80]
n_dir_samples = 15
directions = ['x', 'y', 'z']
dir_colors = {'x': '#1f77b4', 'y': '#ff7f0e', 'z': '#2ca02c'}

dir_results = {d: {w: [] for w in dir_sizes} for d in directions}

print(f"Directional REV sweep: {len(dir_sizes)} sizes x {len(directions)} directions "
      f"x {n_dir_samples} samples = {len(dir_sizes) * len(directions) * n_dir_samples} evaluations...")

t0 = time.time()
for d in directions:
    for w in dir_sizes:
        max_idx = microstructure.shape[0] - w
        for _ in range(n_dir_samples):
            z = np.random.randint(0, max_idx + 1)
            y = np.random.randint(0, max_idx + 1)
            x = np.random.randint(0, max_idx + 1)
            crop = microstructure[z:z+w, y:y+w, x:x+w]

            perc = oi.percolation_check(crop, phase=1, direction=d)
            if perc.percolates:
                res = oi.tortuosity(crop, phase=1, direction=d, solver="flexgmres")
                dir_results[d][w].append(res.tortuosity)

print(f"Completed in {time.time() - t0:.1f}s.")

# --- Plot: CV vs window size for each direction ---
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

for d in directions:
    cv_vals = []
    sz_vals = []
    for w in dir_sizes:
        taus = dir_results[d][w]
        if len(taus) >= 3:
            cv = (np.std(taus) / np.mean(taus)) * 100
            cv_vals.append(cv)
            sz_vals.append(w)
    if sz_vals:
        ax.plot(sz_vals, cv_vals, 'o-', color=dir_colors[d], linewidth=2.5,
                markersize=8, markerfacecolor='white', markeredgewidth=2,
                label=f'{d.upper()}-direction')

ax.axhline(y=5.0, color='r', linestyle='--', linewidth=2, label='5% REV threshold')
ax.set_xlabel("Window Size (Voxels)", fontweight='bold')
ax.set_ylabel("Coefficient of Variation (%)", fontweight='bold')
ax.set_title("Directional REV: Does the REV Size Depend on Transport Direction?", fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

# Print a summary
print("\nDirectional mean tortuosity at the largest window size:")
for d in directions:
    taus = dir_results[d][dir_sizes[-1]]
    if len(taus) >= 2:
        print(f"  {d.upper()}: tau = {np.mean(taus):.4f} +/- {np.std(taus):.4f} "
              f"(n={len(taus)}, CV={100*np.std(taus)/np.mean(taus):.1f}%)")
    else:
        print(f"  {d.upper()}: insufficient data")

## 8. Reporting Uncertainty in Practice

When publishing tortuosity measurements, a bare number like "$\tau = 2.14$" is incomplete. The analysis in this tutorial gives you the tools to report uncertainty rigorously. Below we generate a publication-ready summary table that includes everything a reader needs to evaluate the measurement.

**What to report:**
- **Sample size** (voxels and physical units if the voxel size is known).
- **Number of sub-volumes** evaluated and percolation rate.
- **Mean and 95% bootstrap CI** for the tortuosity in each direction.
- **CV** to indicate whether the sample is above the REV threshold.
- **Normality test result** to justify the choice of error bars (Gaussian vs. bootstrap).

In [ ]:
# --- Publication-ready summary table ---
print("=" * 80)
print("UNCERTAINTY SUMMARY — TORTUOSITY MEASUREMENTS")
print("=" * 80)
print(f"{'Image:':<20s} {data_path}")
print(f"{'Domain size:':<20s} {microstructure.shape}")
print(f"{'Transport phase:':<20s} Phase 1")
print()

# Use the Z-direction results from Section 1 for the main table
print(f"{'Window':>8s}  {'N':>4s}  {'Perc%':>6s}  {'Mean tau':>9s}  "
      f"{'95% Boot CI':>18s}  {'CV%':>6s}  {'Normal?':>8s}  {'REV?':>5s}")
print("-" * 80)

for i, w in enumerate(valid_sizes):
    taus = np.array(plot_data[i])
    n = len(taus)
    r = rev_results[int(w)]
    perc_pct = 100 * r['n_percolating'] / r['n_total'] if r['n_total'] > 0 else 0

    # Bootstrap CI
    boot_samples = rng.choice(taus, size=(B, n), replace=True)
    ci_lo, ci_hi = np.percentile(boot_samples.mean(axis=1), [2.5, 97.5])

    # CV and normality
    cv = 100 * np.std(taus) / np.mean(taus)
    is_rev = "Yes" if cv < 5 else "No"

    if n >= 3:
        _, p_sw = stats.shapiro(taus)
        normal = "Yes" if p_sw >= 0.05 else "No"
    else:
        normal = "N/A"

    print(f"{int(w):>6d}^3  {n:4d}  {perc_pct:5.0f}%  {np.mean(taus):9.4f}  "
          f"[{ci_lo:.4f}, {ci_hi:.4f}]  {cv:5.1f}%  {normal:>8s}  {is_rev:>5s}")

print()
print("Recommended reporting format for the largest REV-valid window size:")
print()

# Find the largest window where CV < 5%
rev_idx = None
for i in range(len(valid_sizes) - 1, -1, -1):
    taus = np.array(plot_data[i])
    cv = 100 * np.std(taus) / np.mean(taus)
    if cv < 5:
        rev_idx = i
        break

if rev_idx is not None:
    w = int(valid_sizes[rev_idx])
    taus = np.array(plot_data[rev_idx])
    boot_samples = rng.choice(taus, size=(B, len(taus)), replace=True)
    ci_lo, ci_hi = np.percentile(boot_samples.mean(axis=1), [2.5, 97.5])
    print(f'  "The Z-direction tortuosity was tau = {np.mean(taus):.3f} '
          f'(95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]),')
    print(f'   measured on {len(taus)} random {w}^3 sub-volumes '
          f'(CV = {100*np.std(taus)/np.mean(taus):.1f}%, '
          f'bootstrap CI, n = {len(taus)})."')
else:
    print("  No window size reached CV < 5%. Consider using a larger sample volume.")

## 9. From Uncertainty to Training Data

The scatter plot in Section 3 revealed a clear physical trend: local geometry (porosity, pore connectivity) controls transport resistance. At small scales, the variation across crops captures a wide range of structure–property relationships.

This observation has a practical implication for machine learning. The same sub-volume sampling strategy used here to quantify uncertainty can also generate labelled training data for surrogate models. Each (morphology, tortuosity) pair is a training sample, and the spread across scales provides natural coverage of the feature space.

This idea is developed further in [Tutorial 5: Surrogate Modelling](05_surrogate_modelling.ipynb), where we train a regression model on OpenImpala-generated data. The REV analysis from this tutorial provides a useful reference: a well-calibrated surrogate should produce prediction intervals that narrow with increasing sample size, mirroring the convergence we observed above.

**Continue the series:**
- [Tutorial 4: Effective Diffusivity and Field Visualisation](04_multiphase_and_fields.ipynb) — Use the cell-problem solver and visualise internal fields.
- [Tutorial 5: Surrogate Modelling](05_surrogate_modelling.ipynb) — Use this sampling approach to train ML models.

---

## References & Further Reading

1. **OpenImpala:** S. Mayner, F. Ciucci, *OpenImpala: open-source computational framework for microstructural analysis of 3D tomography data*, SoftwareX (2024). [GitHub](https://github.com/BASE-Laboratory/OpenImpala)
2. **REV concept:** J. Bear, *Dynamics of Fluids in Porous Media*, Dover Publications (1972). The classical introduction to REV in the context of porous media transport.
3. **REV for electrodes:** J. R. Wilson et al., *Three-dimensional reconstruction of a solid-oxide fuel-cell anode*, Nature Materials 5, 541–544 (2006). [doi:10.1038/nmat1668](https://doi.org/10.1038/nmat1668)
4. **Statistical REV analysis:** A. P. Shearing et al., *Characterization of the 3-dimensional microstructure of a graphite negative electrode from a Li-ion battery*, Electrochemistry Communications 12(3), 374–377 (2010). [doi:10.1016/j.elecom.2009.12.038](https://doi.org/10.1016/j.elecom.2009.12.038)
5. **Coefficient of Variation for REV:** S. J. Cooper et al., *TauFactor: An open-source application for calculating tortuosity factors from tomographic data*, SoftwareX 5, 203–210 (2016). [doi:10.1016/j.softx.2016.09.002](https://doi.org/10.1016/j.softx.2016.09.002)
6. **Bootstrap methods:** B. Efron & R. J. Tibshirani, *An Introduction to the Bootstrap*, Chapman & Hall/CRC (1993). The standard reference for nonparametric confidence intervals.
7. **Normality testing:** S. S. Shapiro & M. B. Wilk, *An analysis of variance test for normality (complete samples)*, Biometrika 52(3–4), 591–611 (1965). [doi:10.1093/biomet/52.3-4.591](https://doi.org/10.1093/biomet/52.3-4.591)
8. **Uncertainty in microstructure characterisation:** P. R. Shearing et al., *3D reconstruction of SOFC anodes using a focused ion beam lift-out technique*, Chemical Engineering Science 64(17), 3928–3933 (2009). [doi:10.1016/j.ces.2009.05.038](https://doi.org/10.1016/j.ces.2009.05.038)
9. **Integral range and geostatistics:** D. Jeulin, *Random Texture Models for Material Structures*, Statistics and Computing 10, 121–132 (2000). [doi:10.1023/A:1008942325749](https://doi.org/10.1023/A:1008942325749). Relates the REV to the integral range of the underlying random field.